# Kaggriculture Match Replay Reproducer

Reproduces a previously played Kaggriculture match from its saved episode JSON -- no agent `.py` files, and no re-running of any agent. Point `REPLAY_JSON_PATH` below at an episode JSON (downloaded from a competition's *Episodes* page, or written out with `json.dump(env.toJSON(), f)`), and this notebook rebuilds the finished `kaggle_environments` `Environment` directly from its recorded `steps` -- same final score, same board history, same replay player as the original run, instantly.

https://www.kaggle.com/code/stpeteishii/kaggriculture-head-to-head-match-runner

https://www.kaggle.com/code/stpeteishii/kaggriculture-match-replay-reproducer

In [ ]:
!pip install -q -U kaggle-environments

import json
from kaggle_environments import make

import kaggle_environments
print("kaggle_environments version:", kaggle_environments.__version__)

## Load a match from its recorded JSON

`load_replay(path)` reads a saved episode JSON and hands its `configuration`, `info`, and `steps` straight to `kaggle_environments.make(...)`. That rebuilds the `Environment` exactly as it stood at the end of the original run -- same farms, same market, same rewards and statuses -- without executing a single agent turn. The game name doesn't need to be set separately; it's read straight out of the JSON's own `"name"` field (`"kaggriculture"`).

In [ ]:
def load_replay(path):
    """
    Reconstruct a finished kaggle_environments Environment from a saved
    episode JSON, instead of running live agents.

    A JSON produced by `json.dump(env.toJSON(), f)` (or downloaded from a
    competition's Episodes page) already contains everything
    `kaggle_environments.make()` needs to rebuild the environment: the
    game name, its configuration, the episode `info` (team names, seed,
    etc.), and the full per-turn `steps` history. Passing `steps=` makes
    `make()` skip running any agent entirely -- the environment comes
    back already DONE, exactly as recorded.
    """
    with open(path) as f:
        data = json.load(f)

    env = make(
        data["name"],
        configuration=data.get("configuration"),
        info=data.get("info"),
        steps=data["steps"],
    )
    return env, data

## Configure -- point this at your episode JSON

If you're running this on Kaggle, attach the JSON as a Notebook Input (**Data -> Upload**, or a Dataset someone else published) and it'll show up under `/kaggle/input/...`. Running locally, just use a normal file path. The file's extension doesn't matter -- only that its contents are the JSON `kaggle_environments` produces for a finished episode.

In [ ]:
REPLAY_JSON_PATH = "/kaggle/input/datasets/kaggle/kaggriculture-episodes-2026-07-30/89006909.json"   # <-- edit me

## Reproduce the match

In [ ]:
env, replay_data = load_replay(REPLAY_JSON_PATH)

names = replay_data.get("info", {}).get("TeamNames", ["Seat 0", "Seat 1"])

final = env.steps[-1]
money_a = final[0].observation.farms[0]["money"]
money_b = final[0].observation.farms[1]["money"]

print(f"{names[0]} (seat 0) final bank: {money_a:,.0f}  -- status: {final[0].status}")
print(f"{names[1]} (seat 1) final bank: {money_b:,.0f}  -- status: {final[1].status}")
print("Winner:", names[0] if money_a > money_b else names[1] if money_b > money_a else "Tie")

## Watch the replay

**Heads up:** `env.render(mode="ipython", ...)` is currently broken for this environment -- it embeds the ~15 MB interactive player via `<iframe srcdoc=...>`, and a naive string-replace in `kaggle_environments` corrupts the bundle's JavaScript, so the cell renders blank. This isn't specific to this notebook -- another user hit the identical blank-render symptom with the plain, documented call (see ["(Resolved) How to show episodes?"](https://www.kaggle.com/competitions/kaggriculture/discussion/732646) -- "resolved" only by switching to someone else's notebook, not by an actual fix).

**The fix:** save the player to its own file and point an `IFrame` at that file *by URL* (`src="./replay.html"`), instead of embedding its content inline (`srcdoc`). Verified working in a real Jupyter Notebook 7 session. This should also render inline on Kaggle, since Kaggle's notebook UI is Jupyter-based -- if it doesn't, `replay.html` is still saved to disk and can be opened directly from the notebook's Output tab.

In [ ]:
from IPython.display import IFrame, display

player_html = env.render(mode="html", width=800, height=800)
with open("replay.html", "w") as f:
    f.write(player_html)

display(IFrame(src="./replay.html", width=800, height=800))